# 1. Read saved Calliope results and inspect the graph

This tutorial reads the supplied national and urban CSV exports from **Calliope 0.7.0**. It does not construct a model or run an optimiser. You will create a graph, query capacities and energy summaries, inspect omitted information, and preserve identifiers across two exports.

Run the cells from top to bottom after installing the notebook dependencies described in [the notebook guide](README.md). Run Jupyter from this repository or one of its subdirectories. Generated files go into `build/notebooks/01_saved_results/`; source files remain unchanged.

In [1]:
from pathlib import Path
import json
from decimal import Decimal

import pandas as pd
from IPython.display import display
from calliomapper import Translator
from calliomapper.queries import run_query
from calliomapper.vocabulary import OC

# Show complete explanations rather than shortening them with ellipses.
pd.set_option("display.max_colwidth", None)

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "examples" / "calliope-0.7.0-glpk").is_dir()
)
EXAMPLES = REPO_ROOT / "examples" / "calliope-0.7.0-glpk"
NATIONAL = EXAMPLES / "national_scale" / "results_directory"
OUT = REPO_ROOT / "build" / "notebooks" / "01_saved_results"
OUT.mkdir(parents=True, exist_ok=True)


def answer_table(result):
    """Keep RDF numeric values and represent unavailable values as None."""
    return pd.DataFrame(
        [[term.toPython() if term is not None else None for term in row] for row in result],
        columns=[str(variable) for variable in result.vars],
    )

## Create the national graph

The default selection includes structure, selected capacities and technical settings, supported energy summaries, and costs. `save()` writes the graph as N-Quads, an identifier manifest, and a report. `translate()` returns the same graph as an RDFLib Dataset for use in Python.

The report distinguishes structural validation from OWL reasoning. Translation runs SHACL checks; it does not run an OWL reasoner or infer additional facts.

In [2]:
national = Translator(NATIONAL)
national_path = OUT / "national.nq"
national.save(national_path)
national_graph = national.translate()

assert national.report["shacl"] == "passed"
display(pd.DataFrame([{
    "SHACL": national.report["shacl"],
    "output series": national.report["external_output_series"],
    "energy summaries": national.report["energy_summaries"],
    "OWL reasoning": national.report["owl_reasoning"],
}]))

,SHACL,output series,energy summaries,OWL reasoning
0,passed,13,13,not executed by translation


## Query calculated flow capacities

The installed-capacity query returns one record for each selected node, technology and carrier. The first eight rows are displayed below; the complete answer remains in `capacities`. Missing unit values are deliberate: the saved numerical array does not, by itself, establish a particular unit such as kW. Tutorial 2 adds an explicitly evidenced interpretation.

In [3]:
capacities = answer_table(run_query(national_graph, "installed_capacity"))
assert len(capacities) == 15
ccgt = capacities.loc[(capacities["node"] == "region1") & (capacities["technology"] == "ccgt")]
assert len(ccgt) == 1
assert ccgt.iloc[0]["value"] == Decimal("30000.0")
assert capacities["unit"].isna().all()
display(capacities[["node", "technology", "carrier", "value", "unit"]].head(8))

,node,technology,carrier,value,unit
0,region1,ccgt,power,30000.0,None
1,region1,demand_power,power,39033.52,None
2,region1,region1_to_region1_1,power,9000.0,None
3,region1,region1_to_region1_2,power,0.0,None
4,region1,region1_to_region1_3,power,2280.38588235294,None
5,region1,region1_to_region2,power,3230.47294117647,None
6,region1_1,csp,power,10000.0,None
7,region1_1,region1_to_region1_1,power,9000.0,None


## Inspect one calculated energy summary

The saved mathematics defines `flow_out` as energy per timestep. The supported summary therefore sums `flow_out × timestep_weights`; it does not multiply duration a second time or annualise the result. CallioMapper records this calculation as a separate activity from the original optimisation.

The value below is checked against the supplied 120-timestep national CSV series. This assertion concerns this fixed example; a different saved execution can produce a different result.

In [4]:
energy = answer_table(run_query(national_graph, "cq03", bindings={
    "kind": OC.AggregatedResultRecord,
    "node": "region1",
    "technology": "ccgt",
    "variable": "flow_out",
}))
assert len(energy) == 1
assert energy.iloc[0]["value"] == Decimal("3411944.78014550320")
display(energy[["node", "technology", "variable", "role", "value", "dimension", "unit"]])

,node,technology,variable,role,value,dimension,unit
0,region1,ccgt,flow_out,summary,3411944.78014550320,energy,None


## Read the urban example and its omissions

The urban example includes additional mathematics. Its capacities, costs and references to output series are supported, but automatic energy aggregation is deliberately withheld. The omission report explains this distinction. An absent summary does not mean that the saved model produced no energy.

In [5]:
urban = Translator(EXAMPLES / "urban_scale" / "results_directory")
urban.save(OUT / "urban.nq")
assert urban.report["external_output_series"] == 21
assert urban.report.get("energy_summaries", 0) == 0

summary_counts = pd.DataFrame([
    {"example": "national", "output series": national.report["external_output_series"],
     "energy summaries": national.report.get("energy_summaries", 0)},
    {"example": "urban", "output series": urban.report["external_output_series"],
     "energy summaries": urban.report.get("energy_summaries", 0)},
])
custom_math_omissions = pd.DataFrame([
    {"variable": item["variable"], "reason": item["reason"]}
    for item in urban.report["omissions"] if "custom mathematics" in item["reason"]
]).drop_duplicates()
assert not custom_math_omissions.empty
display(summary_counts)
display(custom_math_omissions)

,example,output series,energy summaries
0,national,13,13
1,urban,21,0


,variable,reason
0,flow_out,Automatic energy summaries are not supported for custom mathematics or non-base mode.


## Inspect identity and version evidence

Keep the manifest beside the graph. It stores the namespace and fingerprints used to identify the saved configuration and execution. The report separately records how the reader identified the Calliope format.

Stable Calliope normally omits the actual executable-version field from `attrs.yaml`. Its configured version is the intended version, not proof of the executable used. The preparation manifest supplied with these examples independently records their actual Calliope 0.7.0 execution; the translator does not invent that missing source assertion.

In [6]:
manifest_path = national_path.with_suffix(".nq.manifest.json")
manifest = json.loads(manifest_path.read_text())
assert manifest["namespace"] == national.namespace
assert manifest["execution_digest"] == national.source.execution_digest

version_fields = ["calliope_version", "calliope_configured_version", "calliope_format_version", "calliope_version_evidence"]
version_evidence = pd.DataFrame([{"field": key, "value": national.report[key]} for key in version_fields])
display(version_evidence.fillna("Not recorded in the saved export"))
display(pd.DataFrame([{"field": key, "value": manifest[key]} for key in
                      ["namespace", "configuration_digest", "execution_digest"]]))

,field,value
0,calliope_version,Not recorded in the saved export
1,calliope_configured_version,0.7.0
2,calliope_format_version,0.7.0
3,calliope_version_evidence,config.init.calliope_version (intended version; producing version not recorded)


,field,value
0,namespace,urn:uuid:5b4b2823-1b88-4bcc-afa7-ca0bd28bcdff:
1,configuration_digest,1bb2fcc7bced1ce6a46f3f700a430a6445f1bae83c956e5772d5f0edd029c215
2,execution_digest,f55be9a2f969b70b49fee3c5823786434a78da2fc718b60b785f0df3cc88d68d


## Reuse identifiers for a smaller export

Supplying the existing manifest tells CallioMapper that a new selection belongs to the same export namespace. Here the graph contains structure only, while the unchanged configuration and saved execution keep their identifiers. Without this explicit connection, a separate output normally receives an independent namespace.

In [7]:
structure_only = Translator(NATIONAL, groups={"structure"}, identity_file=manifest_path)
structure_only.save(OUT / "national-structure.nq")
checks = {
    "same namespace": structure_only.namespace == national.namespace,
    "same configuration": structure_only.context.configuration == national.context.configuration,
    "same saved execution": structure_only.context.execution == national.context.execution,
    "no capacity results selected": not list(run_query(structure_only.translate(), "installed_capacity")),
}
assert all(checks.values())
display(pd.DataFrame(checks.items(), columns=["check", "passed"]))

,check,passed
0,same namespace,True
1,same configuration,True
2,same saved execution,True
3,no capacity results selected,True


To explore further, change a query binding, inspect more rows in `capacities`, or choose another group in a new `Translator` call. The [usage guide](../docs/usage.md) explains timestep selection and the currently supported variables. Continue with [Tutorial 2](02_supplied_context.ipynb) to see which answers require researcher-supplied information.